In [1]:
import json
from pathlib import Path

import torch
import numpy as np

SEED = 42
MAX_LENGTH = 64
MODEL_NAME = "indobenchmark/indobert-base-p1"

In [2]:
label_map = json.loads(Path("../data/ner_label_map.json").read_text())
label2id = {k: int(v) for k, v in label_map["label2id"].items()}
id2label = {int(k): v for k, v in label_map["id2label"].items()}
LABEL_LIST = [id2label[i] for i in range(len(id2label))]
print(LABEL_LIST)


['O', 'B-ADD_ON', 'I-ADD_ON', 'B-DISH', 'I-DISH', 'B-DRINK', 'I-DRINK', 'B-MODIFIER', 'I-MODIFIER', 'B-QUANTITY', 'I-QUANTITY', 'B-REMOVE', 'I-REMOVE', 'B-SIZE', 'I-SIZE', 'B-TABLE', 'I-TABLE', 'B-TAKE_AWAY', 'I-TAKE_AWAY']


## Step 1 — Load splits

In [3]:
from datasets import Dataset

def load_split(jsonl_path: str) -> Dataset:
    records = []
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            records.append({
                "input_ids": rec["input_ids"],
                "attention_mask": rec["attention_mask"],
                "labels": rec["labels"],
            })
    return Dataset.from_list(records)


train_dataset = load_split("../data/ner_dataset_train.jsonl")
val_dataset   = load_split("../data/ner_dataset_val.jsonl")
test_dataset  = load_split("../data/ner_dataset_test.jsonl")

print(train_dataset)
print(val_dataset)
print(test_dataset)


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4080
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 510
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 510
})


## Step 2 — Class imbalance

Entity tags skew toward `O` and toward common tags (DISH, QUANTITY) vs rare ones (REMOVE, TABLE). Inverse-frequency class weights computed from the **train split only**, same convention as `notebooks/3.2`'s Step 4.

In [4]:
from collections import Counter

def compute_label_counts(jsonl_path: str) -> Counter:
    """Count label ids across a split, excluding -100 (ignored/padding)."""
    counts = Counter()
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            for lbl_id in rec["labels"]:
                if lbl_id != -100:
                    counts[lbl_id] += 1
    return counts


train_label_counts = compute_label_counts("../data/ner_dataset_train.jsonl")
for lbl_id in range(len(LABEL_LIST)):
    print(f"  {id2label[lbl_id]:14s}: {train_label_counts[lbl_id]:6d}")


  O             :  24385
  B-ADD_ON      :    131
  I-ADD_ON      :     65
  B-DISH        :   1464
  I-DISH        :   1651
  B-DRINK       :    474
  I-DRINK       :    586
  B-MODIFIER    :    163
  I-MODIFIER    :     72
  B-QUANTITY    :   1678
  I-QUANTITY    :   1482
  B-REMOVE      :    412
  I-REMOVE      :    313
  B-SIZE        :     72
  I-SIZE        :     21
  B-TABLE       :      3
  I-TABLE       :      5
  B-TAKE_AWAY   :     29
  I-TAKE_AWAY   :      0


In [5]:
total_tokens = sum(train_label_counts.values())
n_classes = len(LABEL_LIST)

class_weights = torch.tensor(
    [total_tokens / (n_classes * max(train_label_counts[i], 1)) for i in range(n_classes)],
    dtype=torch.float,
)

for lbl_id, weight in enumerate(class_weights.tolist()):
    print(f"  {id2label[lbl_id]:14s}: weight={weight:.3f}")

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)


  O             : weight=0.071
  B-ADD_ON      : weight=13.261
  I-ADD_ON      : weight=26.726
  B-DISH        : weight=1.187
  I-DISH        : weight=1.052
  B-DRINK       : weight=3.665
  I-DRINK       : weight=2.964
  B-MODIFIER    : weight=10.657
  I-MODIFIER    : weight=24.127
  B-QUANTITY    : weight=1.035
  I-QUANTITY    : weight=1.172
  B-REMOVE      : weight=4.216
  I-REMOVE      : weight=5.550
  B-SIZE        : weight=24.127
  I-SIZE        : weight=82.722
  B-TABLE       : weight=579.053
  I-TABLE       : weight=347.432
  B-TAKE_AWAY   : weight=59.902
  I-TAKE_AWAY   : weight=1737.158


### Sanity check

In [6]:
assert class_weights.argmin().item() == label2id["O"], "O should get the lowest weight"
assert loss_fn.ignore_index == -100
print("Step 2 sanity checks passed.")


Step 2 sanity checks passed.


## Step 3 — Model architecture

Full fine-tune of `indobenchmark/indobert-base-p2` via `AutoModelForTokenClassification`.

In [7]:
from transformers import AutoModelForTokenClassification

device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"device          : {device}")
print(f"total params    : {n_params:,}")
print(f"trainable params: {n_trainable:,}  (full fine-tune — should equal total)")


[transformers] You passed `num_labels=19` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


device          : mps
total params    : 123,865,363
trainable params: 123,865,363  (full fine-tune — should equal total)


### Sanity check: forward pass on a real batch

In [8]:
sample_batch = train_dataset[:4]
batch_input_ids = torch.tensor(sample_batch["input_ids"], device=device)
batch_attention_mask = torch.tensor(sample_batch["attention_mask"], device=device)
batch_labels = torch.tensor(sample_batch["labels"], device=device)

model.eval()
with torch.no_grad():
    outputs = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask)

logits = outputs.logits
print(f"logits shape: {tuple(logits.shape)}  (expect (4, {MAX_LENGTH}, {len(LABEL_LIST)}))")
assert logits.shape == (4, MAX_LENGTH, len(LABEL_LIST))

loss_fn = loss_fn.to(device)
sample_loss = loss_fn(logits.view(-1, len(LABEL_LIST)), batch_labels.view(-1))
print(f"sample weighted loss: {sample_loss.item():.4f}")

model.train()
print("Step 3 sanity checks passed.")


logits shape: (4, 64, 19)  (expect (4, 64, 19))
sample weighted loss: 3.3807
Step 3 sanity checks passed.


## Step 4 — Training

`WeightedTrainer` overrides `compute_loss` to use Step 2's class-weighted loss instead of `Trainer`'s unweighted default, same as `notebooks/3.2`'s Step 6.

In [9]:
import evaluate

seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    """
    Convert per-token logits/labels back into BIO tag sequences (dropping
    -100 positions) and score with seqeval's entity-level precision/recall/F1.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_seqs, pred_seqs = [], []
    for pred_row, label_row in zip(preds, labels):
        true_seq = [id2label[l] for l in label_row if l != -100]
        pred_seq = [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
        true_seqs.append(true_seq)
        pred_seqs.append(pred_seq)

    results = seqeval_metric.compute(predictions=pred_seqs, references=true_seqs)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


In [10]:
from transformers import Trainer, TrainingArguments

class WeightedTrainer(Trainer):
    """Trainer with class-weighted CrossEntropyLoss (Step 2) instead of the unweighted default."""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        weighted_loss_fn = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device), ignore_index=-100
        )
        loss = weighted_loss_fn(logits.view(-1, len(LABEL_LIST)), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


training_args = TrainingArguments(
    output_dir="../models/indobert-ner-bio",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="epoch",
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
train_result = trainer.train()
train_result.metrics


/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.543039,0.868313,0.455901,0.741414,0.564615,0.864328
2,0.823601,0.843269,0.487430,0.705051,0.576383,0.876595
3,0.548989,0.775426,0.489501,0.753535,0.593477,0.873896
4,0.390955,0.826840,0.463970,0.741414,0.570762,0.860893
5,0.286872,0.840149,0.480469,0.745455,0.584323,0.866290


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': 568.4794,
 'train_samples_per_second': 35.885,
 'train_steps_per_second': 2.243,
 'total_flos': 666409138329600.0,
 'train_loss': 0.7186912925570619,
 'epoch': 5.0}

### Save the fine-tuned model

In [12]:
from transformers import AutoTokenizer

final_model_path = "../models/indobert-ner-bio-final"
trainer.save_model(final_model_path)
AutoTokenizer.from_pretrained(MODEL_NAME).save_pretrained(final_model_path)
print(f"Saved fine-tuned model -> {final_model_path}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned model -> ../models/indobert-ner-bio-final


## Step 5 — Evaluation

Run on the held-out **test** split — not seen during training or `load_best_model_at_end` checkpoint selection — and report per-entity-type precision/recall/F1, not just overall, since rare tags (REMOVE, TABLE) are where this model is most likely to be weak.

In [13]:
test_predictions = trainer.predict(test_dataset)
test_logits, test_label_ids = test_predictions.predictions, test_predictions.label_ids
test_preds = np.argmax(test_logits, axis=-1)

print("overall test metrics:")
for k, v in test_predictions.metrics.items():
    print(f"  {k:25s}: {v}")


/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


overall test metrics:
  test_loss                : 0.665825605392456
  test_precision           : 0.48548812664907653
  test_recall              : 0.7650727650727651
  test_f1                  : 0.5940274414850686
  test_accuracy            : 0.8709441327152964
  test_runtime             : 2.0913
  test_samples_per_second  : 243.868
  test_steps_per_second    : 15.302


In [14]:
test_true_seqs, test_pred_seqs = [], []
for pred_row, label_row in zip(test_preds, test_label_ids):
    true_seq = [id2label[l] for l in label_row if l != -100]
    pred_seq = [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
    test_true_seqs.append(true_seq)
    test_pred_seqs.append(pred_seq)

per_entity_report = seqeval_metric.compute(
    predictions=test_pred_seqs, references=test_true_seqs
)

print("Per-entity-type breakdown (test set):")
for entity_type, scores in per_entity_report.items():
    if entity_type.startswith("overall"):
        continue
    print(
        f"  {entity_type:10s}  precision={scores['precision']:.3f}  "
        f"recall={scores['recall']:.3f}  f1={scores['f1']:.3f}  "
        f"n={scores['number']}"
    )
print()
print(f"  overall precision={per_entity_report['overall_precision']:.3f}  "
      f"recall={per_entity_report['overall_recall']:.3f}  "
      f"f1={per_entity_report['overall_f1']:.3f}  "
      f"accuracy={per_entity_report['overall_accuracy']:.3f}")


Per-entity-type breakdown (test set):
  ADD_ON      precision=0.123  recall=0.714  f1=0.211  n=14
  DISH        precision=0.604  recall=0.767  f1=0.676  n=159
  DRINK       precision=0.592  recall=0.792  f1=0.677  n=53
  MODIFIER    precision=0.167  recall=0.375  f1=0.231  n=24
  QUANTITY    precision=0.750  recall=0.859  f1=0.801  n=192
  REMOVE      precision=0.127  recall=0.500  f1=0.203  n=26
  SIZE        precision=0.200  recall=0.455  f1=0.278  n=11
  TAKE_AWAY   precision=0.667  recall=1.000  f1=0.800  n=2

  overall precision=0.485  recall=0.765  f1=0.594  accuracy=0.871


### Misclassified examples

In [15]:
bio_records_by_id = {}
with open("../data/ner_dataset_bio.jsonl", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        bio_records_by_id[rec["id"]] = rec

test_ids = []
with open("../data/ner_dataset_test.jsonl", encoding="utf-8") as f:
    for line in f:
        test_ids.append(json.loads(line)["id"])

n_mismatch = 0
for rec_id, true_seq, pred_seq in zip(test_ids, test_true_seqs, test_pred_seqs):
    if true_seq != pred_seq:
        n_mismatch += 1
print(f"Mismatched records: {n_mismatch} / {len(test_ids)}\n")

shown = 0
for rec_id, true_seq, pred_seq in zip(test_ids, test_true_seqs, test_pred_seqs):
    if true_seq == pred_seq or shown >= 10:
        continue
    tokens = bio_records_by_id[rec_id]["tokens"]
    print(f"id={rec_id}")
    for tok, t, p in zip(tokens, true_seq, pred_seq):
        marker = "" if t == p else "  <-- MISMATCH"
        print(f"    {tok:15s} true={t:10s} pred={p:10s}{marker}")
    print()
    shown += 1


Mismatched records: 215 / 510

id=2340
    em              true=O          pred=O         
    kurangin        true=O          pred=B-REMOVE    <-- MISMATCH
    ketopraknya     true=B-DISH     pred=B-DISH    
    jadi            true=O          pred=O         
    jadi            true=O          pred=O         
    tiga            true=B-QUANTITY pred=B-QUANTITY
    porsi           true=I-QUANTITY pred=I-QUANTITY
    aja             true=O          pred=O         

id=2672
    mie             true=O          pred=B-DISH      <-- MISMATCH
    mie             true=B-DISH     pred=I-DISH      <-- MISMATCH
    gorengnya       true=I-DISH     pred=I-DISH    
    pedes           true=B-MODIFIER pred=B-MODIFIER
    gak             true=O          pred=O         

id=211
    pe              true=O          pred=O         
    pesan           true=O          pred=O         
    satu            true=B-QUANTITY pred=B-QUANTITY
    porsi           true=I-QUANTITY pred=I-QUANTITY
    nasi          

## Step 6 — Inference

Load the saved model fresh from disk to confirm it's reloadable standalone. `predict_entities(text)` runs raw text through tokenize -> predict -> decode-to-words, then extracts contiguous `B-X`/`I-X` spans per entity type — the structured-order output downstream LangGraph nodes will consume.

In [16]:
from transformers import AutoModelForTokenClassification as _AMFTC, AutoTokenizer as _AT

INFERENCE_MODEL_PATH = "../models/indobert-ner-bio-final"

inference_tokenizer = _AT.from_pretrained(INFERENCE_MODEL_PATH)
inference_model = _AMFTC.from_pretrained(INFERENCE_MODEL_PATH)
inference_model.to(device)
inference_model.eval()

print(f"Loaded model from {INFERENCE_MODEL_PATH}")
print(f"id2label: {inference_model.config.id2label}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded model from ../models/indobert-ner-bio-final
id2label: {0: 'O', 1: 'B-ADD_ON', 2: 'I-ADD_ON', 3: 'B-DISH', 4: 'I-DISH', 5: 'B-DRINK', 6: 'I-DRINK', 7: 'B-MODIFIER', 8: 'I-MODIFIER', 9: 'B-QUANTITY', 10: 'I-QUANTITY', 11: 'B-REMOVE', 12: 'I-REMOVE', 13: 'B-SIZE', 14: 'I-SIZE', 15: 'B-TABLE', 16: 'I-TABLE', 17: 'B-TAKE_AWAY', 18: 'I-TAKE_AWAY'}


In [17]:
def predict_entities(text: str) -> dict:
    """
    Run raw text through the fine-tuned entity tagger.

    Returns:
      tokens : word-level tokens (text.split())
      labels : predicted BIO tag per word (first-subword label only)
      spans  : list of {label: <TAG>, text: str, start: int, end: int}
               (end is exclusive, word indices)
    """
    tokens = text.lower().split()
    if not tokens:
        return {"tokens": [], "labels": [], "spans": []}

    encoding = inference_tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    word_ids = encoding.word_ids()
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        logits = inference_model(**encoding).logits[0]
    pred_ids = logits.argmax(dim=-1).tolist()

    word_labels = [None] * len(tokens)
    prev_word_id = None
    for tok_idx, word_id in enumerate(word_ids):
        if word_id is not None and word_id != prev_word_id:
            word_labels[word_id] = id2label[pred_ids[tok_idx]]
        prev_word_id = word_id
    word_labels = [lbl if lbl is not None else "O" for lbl in word_labels]

    spans = []
    i = 0
    while i < len(tokens):
        lbl = word_labels[i]
        if lbl.startswith("B-"):
            span_type = lbl[2:]
            j = i + 1
            while j < len(tokens) and word_labels[j] == f"I-{span_type}":
                j += 1
            spans.append({
                "label": span_type,
                "text": " ".join(tokens[i:j]),
                "start": i,
                "end": j,
            })
            i = j
        else:
            i += 1

    return {"tokens": tokens, "labels": word_labels, "spans": spans}


### Try it on fresh, unseen sentences

In [18]:
demo_sentences = [
    "saya mau pesan ayam bakar dua porsi pedas banget",
    "minta es teh dua gelas ya bang, jangan pakai gula",
    "tolong nasi goreng satu buat meja lima, dibungkus",
    "add sambal sama extra cheese untuk pasta-nya",
]

for text in demo_sentences:
    result = predict_entities(text)
    print(f"input  : {text}")
    print(f"tagged : {' '.join(f'{t}[{l}]' if l != 'O' else t for t, l in zip(result['tokens'], result['labels']))}")
    print(f"spans  : {result['spans']}")
    print()


input  : saya mau pesan ayam bakar dua porsi pedas banget
tagged : saya mau pesan ayam[B-DISH] bakar[I-DISH] dua[B-QUANTITY] porsi[I-QUANTITY] pedas[B-MODIFIER] banget
spans  : [{'label': 'DISH', 'text': 'ayam bakar', 'start': 3, 'end': 5}, {'label': 'QUANTITY', 'text': 'dua porsi', 'start': 5, 'end': 7}, {'label': 'MODIFIER', 'text': 'pedas', 'start': 7, 'end': 8}]

input  : minta es teh dua gelas ya bang, jangan pakai gula
tagged : minta es[B-DRINK] teh[I-DRINK] dua[B-QUANTITY] gelas[I-QUANTITY] ya bang, jangan[B-REMOVE] pakai[I-REMOVE] gula[B-DRINK]
spans  : [{'label': 'DRINK', 'text': 'es teh', 'start': 1, 'end': 3}, {'label': 'QUANTITY', 'text': 'dua gelas', 'start': 3, 'end': 5}, {'label': 'REMOVE', 'text': 'jangan pakai', 'start': 7, 'end': 9}, {'label': 'DRINK', 'text': 'gula', 'start': 9, 'end': 10}]

input  : tolong nasi goreng satu buat meja lima, dibungkus
tagged : tolong nasi[B-DISH] goreng[I-DISH] satu[B-QUANTITY] buat meja lima,[B-QUANTITY] dibungkus[B-TAKE_AWAY]
spans  